# Milestone 5 lab, notebook 3 of 7: Which day is this run for?

## Where we are

| Notebook | What you do |
|---|---|
| 1. A clock is not enough | Start two steps by the clock, and get yesterday's report |
| 2. Tasks and arrows | Read the DAG, and run it once |
| **3. Which day is this run for?** (you are here) | The logical date, the data interval, and a run you trigger by hand |
| 4. When a task fails | Break the API, then break the data, and watch what Airflow does |
| 5. The days you missed | Run a week with one command. Run one day again |
| 6. Every five minutes | A lab: the same pipeline as a micro-batch |
| 7. The bad morning | The drill: the API is down, then a page comes twice |

In notebook 2 you ran the DAG "for the business day Sunday 7 June", and it set the order system's clock to Monday morning. A run is **for** one day and **happens** on another. Mixing the two is one of the most common mistakes with schedulers.

In [1]:
from lab_helpers import at, MILESTONE_DIR, report_is_right, schedule_warehouse, show_doc, show_runs, start_fresh, table

at.wait_until_ready()
start_fresh()

Airflow has no runs. The warehouse is empty. The order system is back at its first morning.


## Step 1: A run covers a slice of time

The manager's report on Monday morning is about **Sunday**. The run cannot start before Sunday is over. So:

```
Sunday 07:00 ─────────────── the data interval ─────────────── Monday 07:00
     ▲                                                              ▲
     the logical date: the day the run is FOR                       the run STARTS here
```

| Word | Meaning |
|---|---|
| **Data interval** | The slice of time a run covers. Here: 24 hours, from 07:00 to 07:00 |
| **Logical date** | The day a run is for. It is the start of its data interval |

A run starts when its data interval **ends**. That surprises everyone once.

## Step 2: Start a run by hand, for one day

A person can start a run for any day. That is a **trigger**. The command needs the logical date.

```
airflow dags trigger daily_orders --logical-date 2026-06-07T12:00:00+00:00
```

`2026-06-07T12:00:00+00:00` is 07:00 in Arden on Sunday 7 June. `at.trigger` runs that command and waits for the run to finish.

In [2]:
at.trigger("2026-06-07")
show_runs()

,business_day,started_by,state
0,2026-06-07,manual,success


The run is listed under the business day **7 June**. Now look at which morning the pipeline worked on.

In [3]:
con = schedule_warehouse()
table(con, "SELECT run_id, run_date AS the_morning_it_ran_for, rows_landed FROM ops.load_runs")

,run_id,the_morning_it_ran_for,rows_landed
0,2026-06-08_1,2026-06-08,226


The run is **for** 7 June, and it did the work of the **morning of 8 June**. Both are right. They are two different things.

## Step 3: How the DAG knows

The date is not typed anywhere. Airflow fills it in for each run. Find these lines in the DAG file:

In [4]:
dag_file = (MILESTONE_DIR / "dags" / "daily_orders_dag.py").read_text()
print(dag_file[dag_file.index("# Every run is FOR"):dag_file.index("with DAG(")])

# Every run is FOR one business day: its logical date. In templates that day is {{ ds }}.
# The run HAPPENS the morning after, when that day is over. That is the same for a
# scheduled run, a backfill, and a run a person triggers by hand.
BUSINESS_DAY = "{{ ds }}"
RUN_DATE = "{{ macros.ds_add(ds, 1) }}"




`{{ ds }}` is a **template**: a blank that Airflow fills in when the run starts. For this run it became `2026-06-07`. `{{ macros.ds_add(ds, 1) }}` became `2026-06-08`.

## Step 4: Why not just use today's date?

Because the run does not always happen "today". Trigger a second run now, for Monday 8 June. On the wall, it is the same minute as the first run.

In [5]:
at.trigger("2026-06-08")

table(con, "SELECT run_id, run_date AS the_morning_it_ran_for, loaded_at AS when_it_really_ran FROM ops.load_runs ORDER BY run_id")

,run_id,the_morning_it_ran_for,when_it_really_ran
0,2026-06-08_1,2026-06-08,2026-11-21 10:40:02.135480+00:00
1,2026-06-09_1,2026-06-09,2026-11-21 10:40:12.119091+00:00


Two runs, started within moments of each other, did the work of two **different** mornings. `loaded_at` is nearly the same for both. `run_date` is not.

A pipeline that asked the wall clock "what day is it?" would have loaded the same day twice. The rule, from milestone 3 and now with its full name:

> **Use the logical date, never the clock.**

A run for Sunday must do Sunday's work when it runs on Monday at 07:00, on Monday at 11:00 after a retry, or three weeks later in a backfill.

In [6]:
print("The report equals the answer key for the morning of 9 June:", report_is_right("2026-06-09"))
show_runs()

The report equals the answer key for the morning of 9 June: True


,business_day,started_by,state
0,2026-06-07,manual,success
1,2026-06-08,manual,success


## Step 5: The schedule itself

Nobody has to trigger a run. The DAG has a schedule:

In [7]:
print(dag_file[dag_file.index("    # Every day at 07:00"):dag_file.index("    catchup=")])

    # Every day at 07:00 in Arden. Each run covers the 24 hours before it: its data interval.
    # The interval of business day 7 June is 7 June 07:00 to 8 June 07:00, and the run starts when it ends.
    schedule=CronDataIntervalTimetable("0 7 * * *", timezone=ARDEN),
    start_date=datetime(2026, 6, 1, 12, 0),  # business day 1 June. Its run is the morning of 2 June
    end_date=datetime(2026, 6, 17, 12, 0),  # the last business day with a run



| Setting | Meaning |
|---|---|
| `"0 7 * * *"` | The cron fields from notebook 1: every day at 07:00 |
| `timezone="America/Chicago"` | 07:00 on Arden's clock. Not UTC, and not the clock of the server |
| `CronDataIntervalTimetable` | Each run has a data interval: the 24 hours before it starts |
| `start_date`, `end_date` | The first and the last business day that gets a run |

If today were Monday 8 June 2026, Airflow would start the run for 7 June by itself at 07:00. In this lab the calendar is in the past, so the runs are started by a person: by a trigger here, and by a backfill in notebook 5. The run does not know the difference, and that is the point of the logical date.

In [8]:
con.close()
show_doc("HLD.md", "4. Which day is a run for?")

**From `docs/HLD.md`**

> ## 4. Which day is a run for?
> 
> A run that starts on Monday at 07:00 is **for Sunday**. Airflow keeps the two apart.
> 
> *(There is a diagram here. Open `docs/HLD.md` to see it.)*
> 
> | Word | Meaning | For this run |
> |---|---|---|
> | **Data interval** | The slice of time a run covers | Sunday 07:00 to Monday 07:00 |
> | **Logical date** | The day a run is for. It is the start of the data interval | Sunday 7 June |
> | When the run starts | When its data interval is over | Monday 8 June, 07:00 |
> 
> The rule from milestone 3, now with its full name: **use the logical date, never the clock.** A run for Sunday must load Sunday's morning whether it happens on Monday at 07:00, on Monday at 11:00 after a retry, or three weeks later in a backfill.

## New words

| Word | Meaning | Here |
|---|---|---|
| **Data interval** | The slice of time a run covers | Sunday 07:00 to Monday 07:00 |
| **Logical date** | "A logical date is the day the run is about, not the day it runs." In Airflow it is the start of the data interval | Sunday 7 June |
| **Trigger** | Starting a run by hand, for a day you name | `airflow dags trigger daily_orders --logical-date ...` |

The logical date is defined in the D8Loop chapters [Idempotency, Retries & Exactly-Once](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/idempotency-retries-and-exactly-once) and [Incremental Processing & Backfills](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/incremental-processing-and-backfills). "Data interval" is Airflow's own word, from its documentation.

## What you now know

- A run covers a data interval. Its logical date is the start of that interval: the day it is for.
- A run starts when its data interval ends. The run for Sunday starts on Monday morning.
- Templates such as `{{ ds }}` give each run its own date.
- Use the logical date, never the clock. Then a late run and a backfill do the right day's work.

**Say it in an interview:** "Every run has a logical date and a data interval. My daily run for Sunday starts on Monday at 07:00, when Sunday's interval is over. The tasks get the date from the run, through a template, and never from the system clock. That is why a retry four hours late and a backfill three weeks late both load the right day."

Next: [notebook 4](04_when_a_task_fails.ipynb) makes things fail.